# Module 7 - Market Demand Inference

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("Connected to lli_db")

Connected to lli_db


## Q1 - What is the seasonal production pattern by dosage form?


In [2]:
seasonal_dosage_form_query = """
    SELECT
        TO_CHAR(d.full_date, 'YYYY-MM') AS month,
        TO_CHAR(d.full_date, 'Mon') AS month_short,
        p.dosage_form,
        SUM(j.batch_size) AS total_units
    FROM fact_batch_production f
    JOIN dim_job_order j ON f.jo_id = j.jo_id
    JOIN dim_product p ON f.product_id = p.product_id
    JOIN dim_date d ON f.date_id = d.date_id
    WHERE d.year = 2025
    AND f.stage = 'dry_blending'
    GROUP BY
        TO_CHAR(d.full_date, 'YYYY-MM'),
        TO_CHAR(d.full_date, 'Mon'),
        p.dosage_form
    ORDER BY month, p.dosage_form;
"""

df_seasonal_dosage = pd.read_sql_query(seasonal_dosage_form_query, engine)
df_seasonal_dosage

,month,month_short,dosage_form,total_units
0,2025-01,Jan,CAPSULE,5700000.0
1,2025-01,Jan,FILM-COATED TABLET,19485000.0
2,2025-01,Jan,TABLET,1425000.0
3,2025-01,Jan,SUSTAINED-RELEASE TABLET,3500000.0
4,2025-01,Jan,EXTENDED RELEASE TABLET,200000.0
...,...,...,...,...
61,2025-12,Dec,CAPSULE,2800000.0
62,2025-12,Dec,FILM-COATED TABLET,3485000.0
63,2025-12,Dec,TABLET,852500.0
64,2025-12,Dec,SUSTAINED-RELEASE TABLET,400000.0


In [3]:
# Order months correctly
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

df_seasonal_dosage['month_short'] = pd.Categorical(
    df_seasonal_dosage['month_short'],
    categories=month_order,
    ordered=True
)
df_seasonal_dosage = df_seasonal_dosage.sort_values('month_short')

fig = px.line(
    df_seasonal_dosage,
    x='month_short',
    y='total_units',
    color='dosage_form',
    title='Seasonal Production Pattern by Dosage Form (2025)',
    labels={
        'total_units': 'Total Units Produced',
        'month_short': '',
        'dosage_form': 'Dosage Form'
    },
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(marker=dict(size=6))
fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        title='Total Units Produced'
    ),
    legend=dict(
        orientation='v',
        yanchor='top',
        y=1,
        xanchor='left',
        x=1.01,
        font=dict(size=10)
    )
)
fig.show()

## Q1 — Seasonal Production Pattern by Dosage Form
 
**Observation:**
Film-Coated Tablet (orange) is the dominant dosage form throughout the year, with two pronounced peaks — January (19.6M units) and September (19.5M units) — and a notable trough in June (6.5M units). Capsule (green) follows a similar two-peak pattern, surging in September (9.1M units) after a June low. Sustained-Release Tablet (pink) peaks sharply in February (8.6M units) before declining through mid-year. All other dosage forms — Tablet, Bilayer Tablet, Extended Release, Enteric-Coated, and Modified Release — remain relatively flat and low-volume throughout the year.
 
**Insight:**
The dual-peak pattern for Film-Coated Tablet and Capsule — January and September — is the most analytically significant finding in this chart. January production likely reflects year-end order fulfillment carried into Q1, while the September peak aligns with the rainy season in the Philippines, when respiratory and infectious disease incidence typically rises. This drives demand for the types of medications commonly produced as film-coated tablets and capsules — antihypertensives, vitamins, antibiotics, and cough/cold products. The June trough across most dosage forms is consistent with the audit downtime identified in Module 1. Sustained-Release Tablet's February peak suggests its market demand is front-loaded — possibly driven by chronic disease maintenance medications being ordered at the start of the year.
 
**Recommendation:**
Plan Film-Coated Tablet and Capsule inventory builds in Q2 to cushion the September surge. The June trough caused by audit downtime creates a production gap that directly precedes the Q3 demand peak — this is a structural scheduling risk. Pre-positioning finished goods inventory before the audit period begins would reduce the risk of stockouts during the September peak. For Sustained-Release Tablet, investigate whether the February spike reflects client ordering patterns and whether production scheduling can be smoothed to avoid over-concentration in a single month.
 

## Q2 - Which generics are showing production growth across 2025?


In [4]:
production_growth_query = """
    WITH batch_counts AS (
        SELECT
            p.generic_name,
            SUM(CASE WHEN d.month <= 6 THEN 1 ELSE 0 END) AS h1_batches,
            SUM(CASE WHEN d.month > 6 THEN 1 ELSE 0 END) AS h2_batches,
            COUNT(*) AS total_batches
        FROM fact_batch_production f
        JOIN dim_job_order j ON f.jo_id = j.jo_id
        JOIN dim_product p ON f.product_id = p.product_id
        JOIN dim_date d ON f.date_id = d.date_id
        WHERE d.year = 2025
        AND f.stage = 'dry_blending'
        GROUP BY p.generic_name
        HAVING COUNT(*) >= 10
    )
    SELECT
        generic_name,
        h1_batches,
        h2_batches,
        total_batches,
        h2_batches - h1_batches AS growth,
        ROUND(
            (h2_batches - h1_batches) * 100.0 / NULLIF(h1_batches, 0),
            1
        ) AS growth_pct
    FROM batch_counts
    ORDER BY growth_pct DESC
    LIMIT 15;
"""

df_growth = pd.read_sql_query(production_growth_query, engine)

df_growth['generic_short'] = df_growth['generic_name'].apply(
    lambda x: x[:40] + '...' if len(x) > 40 else x
)

df_growth

,generic_name,h1_batches,h2_batches,total_batches,growth,growth_pct,generic_short
0,TELMISARTAN,3,11,14,8,266.7,TELMISARTAN
1,PARACETAMOL,7,17,24,10,142.9,PARACETAMOL
2,ISOSORBIDE MONONITRATE,7,16,23,9,128.6,ISOSORBIDE MONONITRATE
3,CIPROFLOXACIN HCL,4,9,13,5,125.0,CIPROFLOXACIN HCL
4,TERAZOSIN HCL,5,9,14,4,80.0,TERAZOSIN HCL
5,TELMISARTAN + AMLODIPINE (AS BESILATE),12,21,33,9,75.0,TELMISARTAN + AMLODIPINE (AS BESILATE)
6,ASCORBIC ACID (AS SODIUM ASCORBATE) + ZINC (AS...,18,31,49,13,72.2,ASCORBIC ACID (AS SODIUM ASCORBATE) + ZI...
7,GLUCO+CHON+CAL CITRATE+VIT D3,5,8,13,3,60.0,GLUCO+CHON+CAL CITRATE+VIT D3
8,METRONIDAZOLE,4,6,10,2,50.0,METRONIDAZOLE
9,DAPAGLIFLOZIN,12,18,30,6,50.0,DAPAGLIFLOZIN


In [6]:
# Color bars by growth direction
colors = ['steelblue' if g >= 0 else 'lightcoral'
          for g in df_growth['growth_pct']]

fig = px.bar(
    df_growth,
    x='growth_pct',
    y='generic_short',
    orientation='h',
    title='Top 15 Generics by H1 vs H2 Production Growth (2025, min. 10 batches)',
    labels={
        'growth_pct': 'Growth Rate (%)',
        'generic_short': ''
    },
    text='growth_pct',
    color='growth_pct',
    color_continuous_scale='RdYlGn'
)


fig.update_traces(
    textposition='auto',
    texttemplate='%{text}%'
)
fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    xaxis=dict(showgrid=False, title='H1 vs H2 Growth Rate (%)')
)
fig.show()

## Q2 — Top 15 Generics by H1 vs H2 Production Growth
 
**Observation:**
Telmisartan leads by a wide margin at 266.7% growth from H1 to H2 — meaning H2 production was nearly 3.7x higher than H1. Paracetamol (142.9%), Isosorbide Mononitrate (128.6%), and Ciprofloxacin HCL (125%) follow as the next fastest-growing generics. Terazosin HCL (80%), Telmisartan + Amlodipine (75%), and Ascorbic Acid + Zinc (72.2%) round out the high-growth group. All 15 generics shown are positive — no generic in the minimum-10-batch cohort showed declining production from H1 to H2.
 
**Insight:**
Telmisartan's 266.7% H1-to-H2 growth is a strong market demand signal. Telmisartan is an antihypertensive — a chronic disease medication. A near-quadrupling of production in H2 suggests either a new client contract was secured mid-year, a significant increase in existing client orders, or a strategic decision to build inventory ahead of anticipated year-end demand. The presence of both Telmisartan (standalone) and Telmisartan + Amlodipine (combination) in the top 15 reinforces that the antihypertensive category as a whole is experiencing accelerating demand. Ciprofloxacin HCL's 125% growth — an antibiotic — aligns with the rainy season infectious disease pattern inferred from the seasonal analysis. Dapagliflozin (50%) and Sitagliptin Phosphate (16.7%) appearing in this list signals growing diabetes medication demand, consistent with the Philippines' rising diabetes prevalence.
 
**Recommendation:**
Flag Telmisartan, Paracetamol, Isosorbide Mononitrate, and Ciprofloxacin HCL as high-priority products for 2026 capacity planning. Their H2 2025 production acceleration suggests client demand will likely remain elevated or continue growing into 2026. Ensure raw material supply chains for these four generics are secured well in advance. For the antihypertensive and diabetes medication cluster, consider whether dedicated scheduling windows during Q3 would reduce competition for compression and coating machine time with the high-volume Film-Coated Tablet products.


## Q3 - What is the production concentration risk?


In [7]:
concentration_risk_query = """
    WITH generic_output AS (
        SELECT
            p.generic_name,
            SUM(j.batch_size) AS total_units
        FROM fact_batch_production f
        JOIN dim_job_order j ON f.jo_id = j.jo_id
        JOIN dim_product p ON f.product_id = p.product_id
        JOIN dim_date d ON f.date_id = d.date_id
        WHERE d.year = 2025
        AND f.stage = 'dry_blending'
        GROUP BY p.generic_name
    ),
    ranked AS (
        SELECT
            generic_name,
            total_units,
            ROUND(total_units * 100.0 / SUM(total_units) OVER (), 2) AS pct_of_total,
            ROUND(SUM(total_units) OVER (
                ORDER BY total_units DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) * 100.0 / SUM(total_units) OVER (), 2) AS cumulative_pct,
            ROW_NUMBER() OVER (ORDER BY total_units DESC) AS rank
        FROM generic_output
    )
    SELECT
        generic_name,
        total_units,
        pct_of_total,
        cumulative_pct,
        rank
    FROM ranked
    ORDER BY rank
    LIMIT 20;
"""

df_concentration = pd.read_sql_query(concentration_risk_query, engine)

df_concentration['generic_short'] = df_concentration['generic_name'].apply(
    lambda x: x[:35] + '...' if len(x) > 35 else x
)

df_concentration

,generic_name,total_units,pct_of_total,cumulative_pct,rank,generic_short
0,ATORVASTATIN CALCIUM,34880000.0,11.74,11.74,1,ATORVASTATIN CALCIUM
1,BUTAMIRATE CITRATE,25155000.0,8.47,20.21,2,BUTAMIRATE CITRATE
2,ASCORBIC ACID (SODIUM ASCORBATE) + ZINC (AS GL...,24406180.0,8.22,28.43,3,ASCORBIC ACID (SODIUM ASCORBATE) + ...
3,ASCORBIC ACID (AS SODIUM ASCORBATE) + ZINC (AS...,22300000.0,7.51,35.93,4,ASCORBIC ACID (AS SODIUM ASCORBATE)...
4,DAPAGLIFLOZIN,20480000.0,6.89,42.83,5,DAPAGLIFLOZIN
5,ISOSORBIDE MONONITRATE,18600000.0,6.26,49.09,6,ISOSORBIDE MONONITRATE
6,METFORMIN HYDROCHLORIDE,17175000.0,5.78,54.87,7,METFORMIN HYDROCHLORIDE
7,MEFENAMIC ACID,16800000.0,5.66,60.53,8,MEFENAMIC ACID
8,PARACETAMOL + THIAMINE MONONITRATE (VITAMIN B1...,14700000.0,4.95,65.48,9,PARACETAMOL + THIAMINE MONONITRATE ...
9,PARACETAMOL,9995250.0,3.36,68.84,10,PARACETAMOL


In [8]:
fig = go.Figure()

# Bar — individual product share
fig.add_trace(go.Bar(
    x=df_concentration['generic_short'],
    y=df_concentration['pct_of_total'],
    name='% of Total Output',
    marker_color='steelblue',
    text=df_concentration['pct_of_total'],
    texttemplate='%{text}%',
    textposition='outside'
))

# Line — cumulative share (Pareto line)
fig.add_trace(go.Scatter(
    x=df_concentration['generic_short'],
    y=df_concentration['cumulative_pct'],
    name='Cumulative %',
    mode='lines+markers',
    marker=dict(size=6, color='crimson'),
    line=dict(color='crimson', width=2),
    yaxis='y2'
))

# Reference line at 80% cumulative
fig.add_hline(
    y=80,
    line_dash='dash',
    line_color='orange',
    annotation_text='80% of Output',
    annotation_position='top right',
    yref='y2'
)

fig.update_layout(
    title='Production Concentration Risk — Pareto Analysis (Top 20 Generics, 2025)',
    plot_bgcolor='white',
    xaxis=dict(showgrid=False, tickangle=-35, title=''),
    yaxis=dict(
        showgrid=False,
        title='% of Total Output (Individual)',
        range=[0, 20]
    ),
    yaxis2=dict(
        title='Cumulative %',
        overlaying='y',
        side='right',
        range=[0, 105],
        showgrid=False
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()

## Q3 — Production Concentration Risk — Pareto Analysis
 
**Observation:**
Atorvastatin Calcium alone accounts for 11.74% of total facility output — the single largest share of any generic. The top 5 generics (Atorvastatin Calcium, Butamirate Citrate, Ascorbic Acid + Sodium Ascorbate, Ascorbic Acid + Zinc, and Dapagliflozin) collectively represent approximately 43% of total output. The top 10 generics account for roughly 66% of total output. The cumulative line does not cross 80% within the top 20 generics shown — indicating that the facility's output is more evenly distributed than a classic Pareto concentration would suggest, but the top tier still carries meaningful concentration risk.
 
**Insight:**
The fact that the cumulative line has not reached 80% by the 20th generic is actually a moderately healthy sign — it means no single product or small group of products has captured an overwhelming share of production capacity. However, Atorvastatin Calcium at 11.74% is a notable single-product dependency. If a client reduces orders for Atorvastatin Calcium — through supplier switching, product reformulation, or demand contraction — the facility would face a significant volume gap with limited short-term ability to backfill. Butamirate Citrate at 8.47% represents a similar but smaller risk. The presence of two Ascorbic Acid variants (Sodium Ascorbate at 8.22% and Zinc at 7.51%) in the top 4 suggests high client concentration in the vitamin/supplement segment — if one client accounts for both products, the risk is compounded.
 
**Recommendation:**
Present this Pareto chart to management as a portfolio risk assessment tool. The top 5 generics representing 43% of output should be reviewed for client concentration — determine whether each product is supplied to a single client or multiple clients, as single-client products carry higher revenue risk. For Atorvastatin Calcium specifically, assess whether the 11.74% output share is matched by appropriate contractual protections such as long-term supply agreements or minimum order commitments. Strategically, the facility should target growing the output share of mid-tier generics (ranks 6–15) to reduce dependency on the top 5 and improve portfolio resilience.
 








